# ⚡ AegisX — Train on Colab (free T4)

Trains AegisX-Mini from scratch and packages it for **manual** upload to
Hugging Face. No auto-push, no token needed in this notebook.

**Your laptop never trains anything.** Run this notebook in Colab:
`File > Upload notebook`, then `Runtime > Run all`.

## 1. Setup

In [ ]:
!pip install -q torch

import os
import sys
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Optional: mount Google Drive so checkpoints survive session disconnects.
# Set USE_DRIVE = False to skip the Google login and save locally instead.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Skipping Drive mount - checkpoints will be lost if the session disconnects.')

## 2. Get the AegisX code

Clone the repo (or upload your local copy with `aegisx/`, `data/`, `targets/`).

In [ ]:
WORK = '/content/aegisx'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

# Clone the repo; pull updates if already cloned (safe to re-run)
if not os.path.isdir('.git'):
    !git clone https://github.com/FerzDevZ/AegisX.git .
else:
    !git -C . pull --ff-only
print('Repo ready:')
print(os.listdir(WORK))

## 2b. Optional: download MORE training data (recommended)

Pulls large public cybersecurity corpora (OWASP cheat sheets, ASVS, MITRE
ATT&CK) into `data/raw/`. More good text = a smarter model.
Each file downloads individually, so a single failure does not stop the rest.

In [ ]:
# Set to True to download public corpora (internet required, ~1-2 min).
DOWNLOAD_MORE_DATA = True
if DOWNLOAD_MORE_DATA:
    !python scripts/fetch_corpus.py --target-dir data/raw
else:
    print('Skipped corpus download.')

# Corpus size report
total = sum(os.path.getsize(os.path.join('data/raw', f)) for f in os.listdir('data/raw') if f.endswith('.txt'))
print(f'Corpus now: {total/1024:.0f} KB of text')

## 3. Configure training

Default: **from scratch** on the corpus in `data/raw/`.

In [ ]:
# --- training hyperparameters ---
DATA_DIR      = 'data/raw'
OUT_DIR       = '/content/drive/MyDrive/aegisx/checkpoints/aegisx-mini' if USE_DRIVE else '/content/aegisx/checkpoints/aegisx-mini'
VOCAB_SIZE    = 4096
BLOCK_SIZE    = 256
N_LAYER       = 6
N_HEAD        = 6
N_EMBD        = 384
BATCH_SIZE    = 16
GRAD_ACCUM    = 4
MAX_STEPS     = 2000
LR            = 3e-4
WARMUP_STEPS  = 200
EVAL_EVERY    = 200
EARLY_STOP    = 5      # stop after 5 evals without val improvement
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## 4. Train

In [ ]:
!python -m aegisx.train \
    --data {DATA_DIR} \
    --out {OUT_DIR} \
    --vocab-size {VOCAB_SIZE} \
    --block-size {BLOCK_SIZE} \
    --n-layer {N_LAYER} \
    --n-head {N_HEAD} \
    --n-embd {N_EMBD} \
    --batch-size {BATCH_SIZE} \
    --grad-accum {GRAD_ACCUM} \
    --max-steps {MAX_STEPS} \
    --lr {LR} \
    --warmup-steps {WARMUP_STEPS} \
    --eval-every {EVAL_EVERY} \
    --early-stop-patience {EARLY_STOP} \
    --device {DEVICE}

## 5. Quick sanity check

In [ ]:
import os
if os.path.exists(f'{OUT_DIR}/model.pt'):
    !python -m aegisx.chat --model {OUT_DIR}/model.pt --tokenizer {OUT_DIR}/tokenizer.json \
        --prompt "You are AegisX, a cybersecurity assistant. User: how do I enumerate subdomains?\n\nAegisX:" \
        --max-new-tokens 150 --temperature 0.8 --top-k 50
else:
    print('Skipped: model.pt not found - check the training cell output for errors.')

## 6. Package for manual Hugging Face upload

Copies `model.pt` + `tokenizer.json` into a clean export folder and creates
a ZIP you can download. **No auto-push** - you upload it yourself.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

if not os.path.exists(f'{OUT_DIR}/model.pt'):
    print('Skipped: model.pt not found - nothing to export. Train first.')
else:
    EXPORT_DIR = Path('/content/drive/MyDrive/aegisx/export/aegisx-mini') if USE_DRIVE else Path('/content/aegisx/export/aegisx-mini')
    if EXPORT_DIR.exists():
        shutil.rmtree(EXPORT_DIR)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    # 1. Copy model files
    shutil.copy(f'{OUT_DIR}/model.pt', EXPORT_DIR / 'model.pt')
    shutil.copy(f'{OUT_DIR}/tokenizer.json', EXPORT_DIR / 'tokenizer.json')

    # 2. Model card (nice description for your HF page)
    card = """---
license: mit
language:
  - en
  - id
tags:
  - cybersecurity
  - penetration-testing
  - bug-bounty
---

# AegisX-Mini

A lightweight GPT-style decoder-only model trained **from scratch** on public cybersecurity text. Built with the AegisX repo (https://github.com/FerzDevZ/AegisX).

## Files
- `model.pt` - PyTorch weights (load with `aegisx.model.GPT.load`)
- `tokenizer.json` - byte-level BPE tokenizer

## Use it
```python
from aegisx.chat import generate
print(generate('model.pt', 'tokenizer.json', 'You are AegisX, a cybersecurity assistant. User: what is SQL injection?\\n\\nAegisX:'))
```
"""
    (EXPORT_DIR / 'README.md').write_text(card, encoding='utf-8')

    # 3. Zip it for easy download
    zip_path = Path(str(EXPORT_DIR) + '.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(EXPORT_DIR.iterdir()):
            zf.write(f, arcname=f.name)

    print('Export folder:')
    for f in sorted(EXPORT_DIR.iterdir()):
        print(f'  {f.name}  ({f.stat().st_size:,} bytes)')
    print(f'ZIP: {zip_path}')

## 7. Upload to Hugging Face (manual, 2 minutes)

**Download the export**: in the Files sidebar (left), browse to the export
folder (or the `.zip`) and download it. If you mounted Drive, it's at
`MyDrive/aegisx/export/aegisx-mini`.

**Create the model repo**:
1. Go to https://huggingface.co/new - choose **Model**
2. Name it `aegisx-mini` (owner `FerzDevZ`)
3. Create, then open the **Files** tab
4. **Add file > Upload files** - drop in `model.pt` and `tokenizer.json`
   (add `README.md` too - the export includes a ready model card)
5. Commit - done: https://huggingface.co/FerzDevZ/aegisx-mini

**Later, optional - chat UI (Space)**:
1. https://huggingface.co/new-space - SDK **Gradio**, hardware free
2. Upload `hf/space_app.py` (rename to `app.py`), `hf/requirements.txt`,
   and the `aegisx/` package folder
3. Settings > Variables and secrets: `AEGISX_REPO` = `FerzDevZ/aegisx-mini`

> Prefer the CLI later? `python3 hf/push_to_hub.py --repo FerzDevZ/aegisx-mini
> --checkpoint <export-folder>` (needs HF_TOKEN, optional).